In [24]:
import pandas as pd
import numpy as np
import seaborn as sns 
import matplotlib.pyplot as plt
dataset = pd.read_csv("filtered_thyroid_data.csv")
dataset
all_feature_cols = ['Age', 'Gender', 'Hx Radiothreapy', 'Adenopathy', 'Pathology', 'Focality', 'Risk', 'T', 'N', 'M', 'Stage', 'Response', 'Recurred']
continu_cols = ['Age', 'Recurred']
categorical_cols = ['Gender', 'Hx Radiothreapy', 'Adenopathy', 'Pathology', 'Focality', 'Risk', 'T', 'N', 'M', 'Stage', 'Response', 'Recurred']

map_gender = {"M": 0, "F": 1}
map_yes_no = {"No": 0, "Yes": 1}
map_adenopathy = (lambda x: 0 if x == 'No' else 1)
map_pathology = {"Micropapillary": 0, "Papillary": 1, "Follicular": 2, "Hurthel cell": 3}
map_focality = {"Uni-Focal": 0, "Multi-Focal": 1}
map_risk = {"Low": 0, "Intermediate": 1, "High": 2}
map_t = {"T1a": 0, "T1b": 1, "T2": 2, "T3a": 3, "T3b": 4, "T4a": 5, "T4b": 6}
map_n = {"N0": 0, "N1a": 1, "N1b": 2}
map_m = {"M0": 0, "M1": 1}
map_stage = {"I": 0, "II": 1, "III": 2, "IVA": 3, "IVB": 4}
map_response = {"Excellent": 0, "Structural Incomplete": 1, "Indeterminate": 2, "Biochemical Incomplete": 3}

feature_cols = all_feature_cols[0:12]

dataset["Gender"] = dataset["Gender"].map(map_gender)
dataset["Hx Radiothreapy"] = dataset["Hx Radiothreapy"].map(map_yes_no)
dataset["Adenopathy"] = dataset["Adenopathy"].map(map_adenopathy)
dataset["Pathology"] = dataset["Pathology"].map(map_pathology)
dataset["Focality"] = dataset["Focality"].map(map_focality)
dataset["Risk"] = dataset["Risk"].map(map_risk)
dataset["T"] = dataset["T"].map(map_t)
dataset["N"] = dataset["N"].map(map_n)
dataset["M"] = dataset["M"].map(map_m)
dataset["Stage"] = dataset["Stage"].map(map_stage)
dataset["Response"] = dataset["Response"].map(map_response)
dataset["Recurred"] = dataset["Recurred"].map(map_yes_no)

def stratified_split(dataset, target_column, training_size=0.8, random_state=42, frac=1):
    np.random.seed(random_state)
    train_list, test_list = [], []

    for class_value in dataset[target_column].unique():
        class_data = dataset[dataset[target_column] == class_value]
        class_data = class_data.sample(frac=frac, random_state=random_state)

        split_idx = int(len(class_data) * training_size)
        train_list.append(class_data.iloc[:split_idx])
        test_list.append(class_data.iloc[split_idx:])

    train_set = pd.concat(train_list).reset_index(drop=True)
    test_set = pd.concat(test_list).reset_index(drop=True)
    return train_set, test_set

dataset_training, dataset_testing = stratified_split(dataset, "Recurred")

dataset_training['Age'] = dataset_training['Age'].astype(float)

X_train = dataset_training[feature_cols]
X_test = dataset_testing[feature_cols]

y_train = dataset_training["Recurred"]
y_test = dataset_testing["Recurred"]

X_train.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 306 entries, 0 to 305
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Age              306 non-null    float64
 1   Gender           306 non-null    int64  
 2   Hx Radiothreapy  306 non-null    int64  
 3   Adenopathy       306 non-null    int64  
 4   Pathology        306 non-null    int64  
 5   Focality         306 non-null    int64  
 6   Risk             306 non-null    int64  
 7   T                306 non-null    int64  
 8   N                306 non-null    int64  
 9   M                306 non-null    int64  
 10  Stage            306 non-null    int64  
 11  Response         306 non-null    int64  
dtypes: float64(1), int64(11)
memory usage: 28.8 KB


In [25]:
def pca_fit(X, n_components):
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    
    X_mean = np.mean(X, axis=0)
    X_std = np.std(X, axis=0)
    X_std[X_std == 0] = 1 
    
    X_scaled = (X - X_mean) / X_std

    cov = np.cov(X_scaled, rowvar=False)

    if np.ndim(cov) == 0:
        cov = np.array([[cov]])

    eigenvalues, eigenvectors = np.linalg.eigh(cov)

    sorted_indices = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[sorted_indices]
    eigenvectors = eigenvectors[:, sorted_indices]

    components = eigenvectors[:, :n_components]
    X_pca = np.dot(X_scaled, components)
    explained_variance = eigenvalues[:n_components]

    return X_pca, components, X_mean, X_std, explained_variance

def mca_fit(X_cat, n_components):
    X_dummies = pd.get_dummies(X_cat)
    X_matrix = X_dummies.to_numpy()
    
    row_sums = X_matrix.sum(axis=1, keepdims=True)
    col_sums = X_matrix.sum(axis=0, keepdims=True)
    total = X_matrix.sum()
    
    expected = row_sums @ col_sums / total
    Z = (X_matrix - expected) / np.sqrt(expected)

    U, S, VT = np.linalg.svd(Z, full_matrices=False)
    X_mca = U[:, :n_components] * S[:n_components]
    components = VT[:n_components, :]
    
    return X_mca, components, X_dummies.columns

def mca_transform(X_cat, components, dummy_columns):
    X_dummies = pd.get_dummies(X_cat)
    X_dummies = X_dummies.reindex(columns=dummy_columns, fill_value=0)
    
    X_matrix = X_dummies.to_numpy()
    row_sums = X_matrix.sum(axis=1, keepdims=True)
    col_sums = X_matrix.sum(axis=0, keepdims=True)
    total = X_matrix.sum()
    expected = row_sums @ col_sums / total
    Z = (X_matrix - expected) / np.sqrt(expected)
    
    X_proj = Z @ components.T
    return X_proj
def famd_fit(df, n_components):

    float_cols = df.select_dtypes(include = ['float']).columns
    int_cols = df.select_dtypes(include = ['int']).columns

    X_num =df[float_cols].to_numpy()
    X_pca, pca_components, pca_mean, pca_std, _ = pca_fit(X_num, X_num.shape[1])

    X_cat = df[int_cols]
    X_mca, mca_components, dummy_columns = mca_fit(X_cat, min(n_components, 50))

    X_combined = np.hstack([X_pca, X_mca])
    X_famd, famd_components, _, _, _ = pca_fit(X_combined, n_components)

    return {
        'float_cols' : float_cols,
        'int_cols' : int_cols,
        'X_pca' : X_pca,
        'pca_components' : pca_components,
        'pca_mean' : pca_mean,
        'pca_std' : pca_std,
        'X_mca' : X_mca,
        'mca_components' : mca_components,
        'dummy_columns' : dummy_columns,
        'X_famd' : X_famd,
        'famd_components' : famd_components
    }

def famd_transform(df, model):
    X_num = df[model['float_cols']].to_numpy()
    safe_std = np.where(model['pca_std'] == 0, 1, model['pca_std'])
    
    X_scaled = (X_num - model['pca_mean']) / safe_std
    X_pca = np.dot(X_scaled, model['pca_components'])

    X_cat = df[model['int_cols']]
    X_mca = mca_transform(X_cat, model['mca_components'], model['dummy_columns'])

    X_combined = np.hstack([X_pca, X_mca])
    X_famd = np.dot((X_combined - np.mean(X_combined, axis=0)) / np.std(X_combined, axis = 0), model['famd_components'])
    return X_famd
famd_model = famd_fit(X_train, n_components=2)
X_train_famd = famd_model['X_famd']
X_test = famd_transform(X_test, famd_model)

famd_df = pd.DataFrame(X_train_famd, columns=['Dimensi 1', 'Dimensi 2'])
famd_df['target'] = y_train.values

C:\Users\nizam\AppData\Local\Temp\ipykernel_6408\3478955190.py:37: RuntimeWarning: invalid value encountered in divide
  Z = (X_matrix - expected) / np.sqrt(expected)


LinAlgError: SVD did not converge